# Data Processing Pipeline: Nutrition Metrics & MECE Compliance

## Overview
This notebook processes global nutrition data (stunting, wasting, severe wasting) from JME databases and creates a MECE-compliant (Mutually Exclusive & Collectively Exhaustive) dataset for intervention cost analysis.

## Processing Steps

1. **Load Raw Data**: Extract prevalence estimates for three nutrition metrics across demographic groups (gender, residence, wealth quintiles) from UNICEF excel datasets.

2. **Long-Format Consolidation**: Convert wide multi-level headers into useful format with columns: ISO3, Country, Demographic Group, and prevalence estimates.

3. **Multi-Metric Merge**: Combine stunting, wasting, and severe wasting data on country and demographic group keys.

4. **Population Data Integration**: Attach UN World Population Prospects 2024 under-5 population estimates for 2023 by country.

5. **MECE Stratification**: Filter to mutually exclusive & collectively exhaustive strata: **Wealth Quintile (1–5) × Residence (Urban/Rural)** — eliminating overlapping or incomplete demographic breakdowns.

6. **Complete-Case Analysis**: Remove rows with missing values to ensure data integrity.

7. **Count Computation**: 
   - Allocate national population to each MECE stratum based on wealth/residence distribution
   - Convert prevalence percentages to absolute affected counts

8. **Cost Assignment**: Apply a made up rule-based intervention unit costs based on wealth quintile and residence (e.g., poorest quintile + rural areas incur higher costs).

9. **Output**: Save processed dataset to CSV with columns: ISO3, Country, Demographic_group, prevalence estimates, population counts, and intervention costs.

## Key Design Decisions
- **MECE strata**: Ensures no double-counting and complete population coverage for optimization models
- **Rule-based costs**: Wealth and residence modifiers reflect realistic intervention logistics (poorest regions more expensive to reach)
- **Complete-case deletion**: Prioritizes data quality over missing-data imputation

In [ ]:
from pathlib import Path
import pandas as pd

# Input files: one Excel per nutrition metric
data_raw_dir = Path("../data/raw")
files = {
    "stunting":       data_raw_dir / "jme_expand_database_stunting_2025.xlsx",
    "wasting":        data_raw_dir / "jme_expand_database_wasting_2025.xlsx",
    "severe_wasting": data_raw_dir / "jme_expand_database_severe_wasting_2025.xlsx",
}

# Demographic groups we try to read from each workbook
DEMO_GROUPS = [
    "National", "Male", "Female",
    "Urban", "Rural",
    "Wealth Quintile 1", "Wealth Quintile 2", "Wealth Quintile 3",
    "Wealth Quintile 4", "Wealth Quintile 5",
    "Wealth Quintile 1 Urban", "Wealth Quintile 1 Rural",
    "Wealth Quintile 2 Urban", "Wealth Quintile 2 Rural",
    "Wealth Quintile 3 Urban", "Wealth Quintile 3 Rural",
    "Wealth Quintile 4 Urban", "Wealth Quintile 4 Rural",
    "Wealth Quintile 5 Urban", "Wealth Quintile 5 Rural",
]

# Simple rule-based cost modifiers
WEALTH_MOD  = {"Wealth Quintile 1": 30, "Wealth Quintile 2": 20,
               "Wealth Quintile 3": 10, "Wealth Quintile 4": 5, "Wealth Quintile 5": 0}
RESID_MOD   = {"Rural": 15, "Urban": -5}
BASE_COSTS  = {"stunting": 20, "wasting": 25, "severe_wasting": 30}


# ── helpers ──────────────────────────────────────────────────────────────────

def load_metric(file_path, metric):
    # Read the two-level header sheet and drop metadata row
    df = pd.read_excel(file_path, sheet_name="Trend", skiprows=6, header=[0, 1])
    df = df.iloc[1:].reset_index(drop=True)

    # Keep country identifiers in flat columns for easy joins
    iso_col     = next(c for c in df.columns if c[0] == "ISO")
    country_col = next(c for c in df.columns if c[0] == "Countries and areas")
    ids = df[[iso_col, country_col]].copy()
    ids.columns = ["ISO3", "Country"]

    # Build a tidy long table: one row = country + demographic group
    available = set(df.columns.get_level_values(0))
    rows = []
    for demo in DEMO_GROUPS:
        if demo not in available:
            continue
        col = (demo, "Point Estimate")
        if col not in df.columns:
            continue
        values = df[col].values
        for i, val in enumerate(values):
            try:
                rows.append({"ISO3": ids.iloc[i]["ISO3"], "Country": ids.iloc[i]["Country"],
                             "Demographic_group": demo, f"Point_estimate_{metric}": float(val)})
            except (ValueError, TypeError):
                # Skip non-numeric or missing raw cells
                pass

    return pd.DataFrame(rows)


def mece_population(demo, total_pop):
    # Split population by residence share, then by wealth quintile
    rural_share = 0.40 if "Rural" in demo else 0.60
    return total_pop * rural_share / 5


def intervention_cost(demo, metric):
    # Cost = base metric cost + wealth modifier + residence modifier
    w = next((v for k, v in WEALTH_MOD.items() if k in demo), 0)
    r = next((v for k, v in RESID_MOD.items() if k in demo), 0)
    return BASE_COSTS[metric] + w + r


# ── pipeline ─────────────────────────────────────────────────────────────────

# Extract each metric and merge into one table
group_cols = ["ISO3", "Country", "Demographic_group"]

df_stunt  = load_metric(files["stunting"],       "stunting")
df_wast   = load_metric(files["wasting"],        "wasting")
df_severe = load_metric(files["severe_wasting"], "severe_wasting")

df = (df_stunt
      .merge(df_wast,   on=group_cols)
      .merge(df_severe, on=group_cols))

# Collapse possible duplicates after merge
value_cols = ["Point_estimate_stunting", "Point_estimate_wasting", "Point_estimate_severe_wasting"]
df = df.groupby(group_cols)[value_cols].mean().reset_index()

# Attach UN under-5 population for 2023
pop_file = Path("../data/raw/WPP2024_POP_F02_1_POPULATION_5-YEAR_AGE_GROUPS_BOTH_SEXES.xlsx")
pop = pd.read_excel(pop_file, skiprows=16)
pop = (pop[(pop["Type"] == "Country/Area") & (pop["Year"] == 2023)]
       [["ISO3 Alpha-code", "0-4"]]
       .rename(columns={"ISO3 Alpha-code": "ISO3", "0-4": "Population_u5"}))
pop["Population_u5"] = pd.to_numeric(pop["Population_u5"], errors="coerce")
pop["Population_u5"] = (pop["Population_u5"] * 1000).round().astype("Int64")

df = df.merge(pop, on="ISO3", how="left")

# Keep only MECE strata: wealth quintile x residence
is_mece = df["Demographic_group"].apply(
    lambda g: any(f"Wealth Quintile {i}" in g for i in range(1, 6))
              and ("Urban" in g or "Rural" in g)
)
df = df[is_mece].reset_index(drop=True)

# Complete-case policy: drop rows with any missing field
df = df.dropna().reset_index(drop=True)

# Compute adjusted under-5 population for each MECE row
df["Population_u5_adjusted"] = (
    df.apply(lambda r: mece_population(r["Demographic_group"], r["Population_u5"]), axis=1)
    .round().astype("Int64")
)

# Convert prevalence (%) into absolute counts
for metric in ["stunting", "wasting", "severe_wasting"]:
    df[f"Count_{metric}"] = (
        (df[f"Point_estimate_{metric}"] / 100 * df["Population_u5_adjusted"])
        .round().astype("Int64")
    )

# Add intervention unit costs
for metric in ["stunting", "wasting", "severe_wasting"]:
    df[f"Cost_{metric}"] = df["Demographic_group"].apply(
        lambda g, m=metric: intervention_cost(g, m)
    )

# Save the processed dataset into a CSV file

out = Path("../data/processed/master_df_mece_compliant.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)


print(f"Saved {len(df)} rows × {df.shape[1]} columns → {out}")
print("=" * 266)
print("Sample of processed data:")
print(df.head(3).to_string())
print("=" * 266)

Saved 1131 rows × 14 columns → ..\data\processed\master_df_mece_compliant.csv
Sample of processed data:
  ISO3      Country        Demographic_group  Point_estimate_stunting  Point_estimate_wasting  Point_estimate_severe_wasting  Population_u5  Population_u5_adjusted  Count_stunting  Count_wasting  Count_severe_wasting  Cost_stunting  Cost_wasting  Cost_severe_wasting
0  AFG  Afghanistan  Wealth Quintile 1 Rural                     49.3                     5.4                            1.3        6656181                  532494          262520          28755                  6922             65            70                   75
1  AFG  Afghanistan  Wealth Quintile 1 Urban                     68.9                     4.4                            2.7        6656181                  798742          550333          35145                 21566             45            50                   55
2  AFG  Afghanistan  Wealth Quintile 2 Rural                     41.2                     5.5  